Knowledge

For detecting the crop type -
On USDA website we are downloading the cropland data layer (CDL) which is a raster(dividing image into cells in a grid) with crop labels. It is 30m resolution which aligns with some sentinel-2 bands.
Each pixel in sentinel-2 imagery (after some processing) would correspond to a CDL pixcel indicating the crop type. The model will learn to predict these labels.

For detecting the field boundaries -
CDL has per-pixel crop labels, but fields are contiguous. It is more straight forward to use vector boundaries from the crop sequence boundaries because they are the outlines of fields.

1. download and alignt sentinel-2 data with USDA data (overlap) - same coordinate system, same spatial resolution, same regions, same time periods. sentinel-2 has bands with 10, 20, 60m resolutions. CDL is 30m. We need to resample sentinel-2 bands to 30m to match CDL.
(
to prevent the model only working for specific region and slow to update, we will use both CDL and vector boundaries as training data,
CDL raster: use it as segmentation mask, each pixel represents a crop type. but for field boundaries use a binary mask (field 1 vs. non-field 0)
crop sequence boundaries: polygons of fields. convert these into raster masks where each field is a separate ploygon with unique ID. This can be used for semantic segamentation(the category of a thing, field in this case) or instance segmentation (the specification of things in category like field1, 2, 3...)
)

2. Training data creation - For each sentinel-2 image, create mask from USDA data. With CDL, mask is crop type per pixel. With vector boundaries, rasterize polygons to create binary mask (field 1 vs. non-field 0) or instance masks.

3. Model training - use supervised learning model like U-Net or similiar CNN for semantic segmentation. Inputs are sentinel-2 bands (all 13 or relevant ones like RGB, NIR, SWIR). Output is the predicted mask (field, non-field)

4. Post processing - convert predicted segmentation mask into vector boundaries with polygonization. Calculate acreage by computing the area of each ploygon in vector data. 

5. Validation - compare predicted boundaries with USDA vector boundaries with metrics like IoU (intersection over union) for semantic segmentation, or precision/recall for boundary detection. for acreage, calculate difference between predicted and ground truth areas.

Proposed Plan -

Based on what i have been reading i think what we'd need to do is -
 
Raster image is basically image in a grid (attachement 1)
 
The sentinel-2 data is already distributed as raster imagery. The CDL data we have here is also raster, each pixel is labeled with crop code (like 1 is soybean 2 is wheat), we need to convert the raster into binary (1 field, 0 non-field) mask with the crop field as information (attachment 2).
 
Then, we can derive the vector boundaries from CDL raster to calculate the acreage of field (attachment 3)
 
After we got the target labels (from above which is the binary values), the input data is the sentinel-2 data. We align the sentinel-2 and CDL images by rotating, using same resolution, using same geo location.
 
Then we can split the data into train and test sets, train supervised model on training set with like U-Net algorithm against the target labels, then we use the trained model to predict the field boundaries and calculate the acreage on the test set, generate accuracy etc.
 
after everything we can apply crop type detection like attachment 4

POC Steps -

Import needed libraries


In [3]:
# AWS API related
import boto3
from botocore import UNSIGNED
from botocore.config import Config
# Work with TIFF format files
import rasterio
# Visualization
import matplotlib.pyplot as plt
# Data manipulation
import numpy as np

Download and examine data from both sources but just one county


In [5]:
# Establish unsigned connection to AWS S3
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
# Define bucket name
bucket = 'sentinel-cogs'

Get the bands and visualize them. The generated bands visualization is a file called band_subset.png

In [ ]:
# Bands can be accessed with command like: aws s3 ls --no-sign-request 
# s3://sentinel-cogs/sentinel-s2-l2a-cogs/9/D/VA
# /2021/3/S2B_9DVA_20210323_1_L2A/
base_path = "sentinel-s2-l2a-cogs/9/D/VA/2021/3/S2B_9DVA_20210323_1_L2A/"

# List of bands we are interested in for example
# Blue, Green, Red, NIR  
# Standard Sentinel-2 bands for vegetation analysis (e.g., NDVI)
bands = ['B02', 'B03', 'B04', 'B08']

# Dictorionary to store band data
band_data = {}

for band in bands:
    # Open raster files directly from S3 bucket
    with rasterio.open(f's3://{bucket}/{base_path}/{band}.tif') as src:
        # Read image data
        band_data[band] = src.read(1)
        if 'transform' not in band_data:
            band_data['transform'] = src.transform
            band_data['crs'] = src.crs

# print(band_data)
# In the print, we can see each band is 
# 2D raster (matrix) with shape (10980, 10980)
# pixel values are stored in array. Transform here basically means 10 meters. 
# CRS is the coordinate reference system used
# {'B02': array([[8568, 8480, 8552, ..., 7128, 7076, 7128],
#    [8568, 8552, 8600, ..., 7108, 7152, 7172],
#    [8616, 8472, 8512, ..., 7184, 7208, 7144],
#    ...,
#    [6796, 6820, 6888, ..., 8712, 8680, 8672],
#    [6808, 6836, 6864, ..., 8656, 8576, 8680],
#    [6732, 6808, 6836, ..., 8736, 8656, 8736]], 
#     dtype=uint16), 
#      'transform': Affine(10.0, 0.0, 399960.0,
#    0.0, -10.0, 2100040.0), 
#       'crs': CRS.from_epsg(32709), ..... }

# Visualize the bands
bands_to_plot = ['B02', 'B03', 'B04', 'B08']
reflectance_data = {band: band_data[band].astype(float)/10000 for band in bands_to_plot}

def plot_subset(window_size=1000):
    plt.figure(figsize=(15, 10))
    
    # Take first NxN pixels from each band
    for i, (band, data) in enumerate(reflectance_data.items()):
        plt.subplot(2, 2, i+1)
        subset = data[:window_size, :window_size]
        plt.imshow(subset, cmap='viridis', vmin=0, vmax=1)
        plt.title(f'Band {band} ({["Blue", "Green", "Red", "NIR"][i]})\nFirst {window_size}x{window_size} pixels')
        plt.colorbar(label='Reflectance')
        plt.axis('off')

    plt.savefig('band_subset.png', bbox_inches='tight', dpi=100)
    plt.close()

plot_subset(window_size=1000)  # Start with 1000x1000 pixels